# PID Loop Simulator
## Run a closed loop PID simulator to allow testing different options.
## This version extracts the open loop reproduction of data from a night to use for testing.
## I have also added multiple options for tracking the integral terms.
Craig Lage 28-Apr-26

In [ ]:
import numpy as np
import pickle as pkl
import pandas as pd
import galsim
from collections import deque
import copy
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
from lsst.ts.ofc import OFC, OFCData, StateEstimator, SensitivityMatrix

In [ ]:
class PIDLoopSimulation:
    """
    Closed-loop PID simulator that uses an open-loop reproduction (OLR)
    of nightly Zernike data to test different PID control strategies for
    the Rubin Observatory active optics system.
    """

    def __init__(
        self,
        ofc,
        ofc_data,
        state_estimator,
        sens_mat,
        kp,
        ki,
        kd,
        day_obs=20260420,
        seq_start=97,
        seq_end=147,
        n_correct=3,
        discard_intermediates=False,
        smith_corrector=True,
        control_vmodes=False,
        use_ofc_integral=True,
        max_integral=np.zeros(50),
        control_zernikes=False,
        leaky_integral_factor=1.0,
        use_kalman=False,
        kalman_r_sigma=0.05,
        kalman_q_dof_sigma=1e-6,
        kalman_R_matrix=None,
        kalman_r_floor=1e-6,
        kalman_alpha=1.0,
        kalman_burnin=20,
    ):
        """
        Initialise the simulation and set all configuration options.

        Parameters
        ----------
        ofc : OFC
            Optical Feedback Controller instance (already configured).
        ofc_data : OFCData
            OFC data / configuration object.
        state_estimator : StateEstimator
            State estimator used for vmode calculations.
        sens_mat : np.ndarray
            Sensitivity matrix (shape: [n_sensors * n_zernikes, n_dof]).
        kp : np.ndarray
            Proportional gain array.  Shape (50,) for DOF/vmode control,
            or (4, 23) when control_zernikes=True.
        ki : np.ndarray
            Integral gain array.  Shape (50,) for DOF/vmode control,
            or (4, 23) when control_zernikes=True.
        kd : np.ndarray, shape (50,)
            Derivative gain array for all 50 DOFs.
        day_obs : int
            Observation day in YYYYMMDD format.
        seq_start : int
            First sequence number to include in the simulation.
        seq_end : int
            Last sequence number (exclusive) to include.
        n_correct : int
            Correction lag — how many exposures between applied corrections
            when discard_intermediates is True.
        discard_intermediates : bool
            If True, only apply a correction every n_correct steps.
            If False, apply corrections every step (optionally with Smith predictor).
        smith_corrector : bool
            When discard_intermediates is False, enable the Smith predictor
            correction to compensate for the control lag.
        control_vmodes : bool
            If True, operate in virtual-mode (vmode) space.
            If False, operate in degree-of-freedom (DOF) space.
        use_ofc_integral : bool
            If True, use the integral term as implemented inside the OFC
            controller, clamped by max_integral.
            If False, maintain the integral externally (see control_zernikes).
        max_integral : np.ndarray, shape (50,)
            Maximum absolute value the integral term is allowed to reach.
            Only used when use_ofc_integral=True.
        control_zernikes : bool
            If True, apply Kp and Ki in Zernike space (requires kp and ki
            to have shape (4, 23)) before passing to the OFC.
            If False, control in DOF space (or vmode space).
        leaky_integral_factor : float
            Factor by which integrals are deweighted moving to the past.
            Factor of 1.0 means no deweighting. Older terms in the integral
            are deweighted by leaky_integral_factor^N.
            Only used when use_ofc_integral=False.
        use_kalman : bool
            If True, apply a Kalman filter to the simulated Zernike
            measurements before passing them to getTweak.
            If False (default), Zernikes are passed through unchanged.
        kalman_r_sigma : float
            Standard deviation of Zernike measurement noise (microns).
            Used to build R = kalman_r_sigma² × I₈₄ when kalman_R_matrix
            is None.  Default 0.05 µm.  Ignored if kalman_R_matrix is
            provided.
        kalman_q_dof_sigma : float
            Standard deviation of process noise on the DoF state vector.
            Sets Q = kalman_q_dof_sigma² × I₂₂.  Default 1e-6.
        kalman_R_matrix : np.ndarray, shape (84, 84), or None
            Full measurement noise covariance matrix.  If provided, it is
            used directly as R and kalman_r_sigma is ignored.  Use this
            when you have a calibrated per-mode noise estimate (e.g. from
            the Bo Xin covariance matrix).  Default None.
        kalman_r_floor : float
            Minimum value applied to the diagonal of R after construction.
            Prevents R from being singular (which causes the Kalman gain
            to diverge).  Any diagonal entry smaller than kalman_r_floor
            is raised to this value.  Default 1e-6.
        kalman_alpha : float
            Leaky state-transition factor (0 < alpha <= 1.0).  F = alpha * I₂₂
            so the predict step shrinks the tweak estimate slightly each step:
            x_pred = alpha * x.  This provides a soft restoring force toward
            zero, preventing unbounded drift for near-zero DoFs.  alpha = 1.0
            (default) gives the original non-leaky identity transition.
            Values between 0.8 and 0.99 are a reasonable range to explore.
        kalman_burnin : int
            Number of initial steps during which kalman_x is pinned to the
            actual OFC tweak after each update, allowing the error covariance
            P to converge before free filtering begins.  During burnin the
            Zernikes passed to getTweak are unfiltered.  After burnin, normal
            Kalman filtering takes over.  Default 20.
        """
        # ── External objects ──────────────────────────────────────────────
        self.ofc = ofc
        self.ofc_data = ofc_data
        self.state_estimator = state_estimator
        self.sens_mat = sens_mat

        # ── Data selection ────────────────────────────────────────────────
        self.day_obs = day_obs
        self.seq_start = seq_start
        self.seq_end = seq_end

        # ── Correction options ────────────────────────────────────────────
        self.n_correct = n_correct
        self.discard_intermediates = discard_intermediates
        self.smith_corrector = smith_corrector  # only used when discard_intermediates=False
        self.control_vmodes = control_vmodes

        # ── DOF index helpers ─────────────────────────────────────────────
        # Collapses the full 50-DOF space down to the 22 DOFs we are using.
        # If control_vmodes=True the _used arrays should have 12 components;
        # if control_vmodes=False they should have 22 components.
        self.indices = list(range(0, 17)) + list(range(30, 35))

        # ── PID gains and integrals (passed in from outside the class) ────
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.max_integral = max_integral
        self.use_ofc_integral = use_ofc_integral
        self.control_zernikes = control_zernikes
        self.leaky_integral_factor = leaky_integral_factor

        # ── Kalman filter parameters ────────────────────────────────────────────
        self.use_kalman = use_kalman
        # R: measurement noise covariance, shape (84, 84)
        # Use the supplied matrix if given, otherwise build from kalman_r_sigma.
        if kalman_R_matrix is not None:
            self.kalman_R = kalman_R_matrix.copy()
        else:
            self.kalman_R = (kalman_r_sigma ** 2) * np.eye(84)
        # Floor the diagonal of R to prevent singularity.
        # Zero or near-zero diagonal entries cause the Kalman gain to
        # diverge, so we ensure every mode has at least kalman_r_floor variance.
        diag_idx = np.diag_indices(84)
        self.kalman_R[diag_idx] = np.maximum(self.kalman_R[diag_idx], kalman_r_floor)
        self.kalman_alpha = kalman_alpha
        self.kalman_burnin = kalman_burnin
        self._kalman_burnin_init = kalman_burnin  # saved to reset each run
        # Q: process noise covariance on the DoF state, shape (22, 22)
        self.kalman_Q = (kalman_q_dof_sigma ** 2) * np.eye(22)
        # Runtime state (initialised by initKalmanState inside runSimulation)
        self.kalman_x = None   # tweak estimate, shape (22,)
        self.kalman_P = None   # error covariance, shape (22, 22)

        # ── Plot / label helpers ──────────────────────────────────────────
        self.zk_groups = [[0], [11 - 4], [1, 2], [3, 4], [5, 6], [22]]
        self.zk_group_labels = ['Z4', 'Z11', 'Z5 / Z6', 'Z7 / Z8', ' Z9 / Z10', 'Z22']

        self.groups = [[0], [5], [1, 2], [6, 7], [3, 4], [8, 9]]
        self.group_labels = [
            'M2 dz\n[um]', 'Cam dz\n[um]', 'M2 dx/dy\n[um]',
            'Cam dx/dy\n[um]', 'M2 tilts\n[arcsec]', 'Cam tilts\n[arcsec]',
        ]
        self.labels = [
            'm2 dz', 'm2 dx', 'm2 dy', 'm2 rx', 'm2 ry',
            'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
        ]

        self.mirror_groups = [
            [10, 11, 30, 31], [12, 34], [13, 14, 32, 33],
            [15, 16, 35, 36], [17, 18, 37, 38],
        ]
        self.mirror_group_labels = ['Astig', 'Spherical', 'Trefoil', 'Coma', 'Quad']

        self.all_labels = [
            'M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
            'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
            'B1,1', 'B1,2', 'B1,3', 'B1,4', 'B1,5',
            'B1,6', 'B1,7', 'B1,8', 'B1,9', 'B1,10',
            'B1,11', 'B1,12', 'B1,13', 'B1,14', 'B1,15',
            'B1,16', 'B1,17', 'B1,18', 'B1,19', 'B1,20',
            'B2,1', 'B2,2', 'B2,3', 'B2,4', 'B2,5',
            'B2,6', 'B2,7', 'B2,8', 'B2,9', 'B2,10',
            'B2,11', 'B2,12', 'B2,13', 'B2,14', 'B2,15',
            'B2,16', 'B2,17', 'B2,18', 'B2,19', 'B2,20',
        ]

        self.vmode_labels = [
            'Vmode1\nM2 tilts -rx-ry', 'Vmode2\nM2 tilts -rx+ry',
            'Vmode3\nCam tilts -rx+ry', 'Vmode4\nCam tilts rx+ry',
            'Vmode5\nZ4-Focus', 'Vmode6\nZ5-Astig-Oblique',
            'Vmode7\nZ6-Astig-Vert', 'Vmode8\nZ7-Coma-Vert',
            'Vmode9\nZ8-Coma-Horiz', 'Vmode10\nZ9-Trefoil-Vert',
            'Vmode11\nZ10-Trefoil-Oblique', 'Vmode12\nZ11-Spherical',
        ]

        # Corner detector names used for FWHM intrinsic subtraction
        self.corner_detnames = ["R00_SW0", "R04_SW0", "R40_SW0", "R44_SW0"]

        # ── Simulation outputs (populated by runSimulation) ───────────────
        self.seqs = None
        self.z_measured = None
        self.z_opens = None
        self.rots = None
        self.filters = None
        self.new_zernikes = None
        self.integrals = None
        self.tweaks = None
        self.trims = None
        self.these_seqs = None
        self.xs = None # Kalman x vectors

    # ─────────────────────────────────────────────────────────────────────
    # Helper methods
    # ─────────────────────────────────────────────────────────────────────

    def applyTrim(self, zernikes, trim, subtract=True):
        """
        Apply (or remove) a DOF trim vector to a set of Zernike measurements
        using the sensitivity matrix. The trim is always in DoF space
        even when controlling in Vmode space

        Parameters
        ----------
        zernikes : np.ndarray, shape (4, 23)
            Measured Zernike coefficients at the four corner detectors.
        trim : np.ndarray, shape (22,)
            DOF trim vector (22 active DOFs).
        subtract : bool
            If True, subtract the trim contribution from the Zernikes
            (forward simulation step).
            If False, add it back (used to construct the open-loop reproduction).

        Returns
        -------
        np.ndarray, shape (4, 23)
            Zernike array with the trim contribution applied.
        """
        zernikes_change = self.sens_mat @ trim
        zernikes_change = zernikes_change.reshape(4, 21)
        # Zero out Z20 and Z21 (indices 16 and 17 after reshape)
        zernikes_change = np.insert(zernikes_change, obj=16, values=0, axis=1)
        zernikes_change = np.insert(zernikes_change, obj=16, values=0, axis=1)
        if subtract:
            return zernikes - zernikes_change
        else:
            # Removes the trim to reconstruct the open-loop Zernikes.
            return zernikes + zernikes_change

    def evaluateIntegralTerm(self):
        """
        Evaluate the integral accumulator term used when
        use_ofc_integral=False.

        When use_ofc_integral=False the integral term is a list going into the past.
        The older terms are deweigthed by leaky_integral_factor.

        * control_zernikes=True  → each element is np.ndarray (4, 23),
          one Zernike frame per detector.
        * control_zernikes=False → each element is np.ndarray (22,),
          one value per active DOF, or np.ndarray(12,) if control_vmodes=True.

        Returns: The weighted sum of the integral list
        """
        if self.use_ofc_integral:
            return None
        if len(self.integral_terms) == 0:
            if self.control_zernikes:
                return np.zeros([4,23])
            else:
                if self.control_vmodes:
                    return np.zeros(12)
                else:
                    return np.zeros(22)
        weighted_sum = np.zeros_like(self.integral_terms[-1])
        for i, term in enumerate(reversed(self.integral_terms)):
            weighted_sum += self.leaky_integral_factor**i * term
        return weighted_sum


    # ─────────────────────────────────────────────────────────────────────
    # Kalman filter methods
    # ─────────────────────────────────────────────────────────────────────

    def buildKalmanMatrices(self):
        """
        Construct the constant Kalman filter matrices and store them as
        instance attributes.

        The state vector x is the current per-visit DoF correction (tweak)::

            x = tweak  (22,)   per-visit DoF correction being estimated

        The tweak is small and of the same order as the Zernike residuals
        scaled by the sensitivity matrix, making it a well-conditioned
        estimation target.  The filter answers the question: given the noisy
        Zernike residuals, what is the best estimate of the DoF correction
        implied by this measurement?

        Using the tweak (rather than the accumulated trim) avoids the
        divergence that arises when S @ trim produces Zernike predictions
        far larger than the small residuals actually observed.

        Matrices built
        --------------
        self.kalman_F  : (22, 22) state transition  — leaky identity
            F = alpha * I₂₂  (alpha = self.kalman_alpha)
            The tweak estimate decays slightly each step, providing a
            soft restoring force that prevents unbounded drift.
            When alpha=1.0 this is the standard identity transition.

        self.kalman_B  : (22, 22) control-input matrix  — zeros
            B = 0₂₂
            There is no separate control input; the tweak is the state
            being estimated, not something added from outside.

        self.kalman_H  : (84, 22) observation matrix  — sensitivity matrix
            H = S  (sens_mat, shape 84×22)
            The predicted Zernike residuals are H @ x = S @ tweak.

        Side effects
        ------------
        Sets self.kalman_F, self.kalman_B, self.kalman_H.
        """
        # F: leaky identity — tweak estimate decays slightly each step.
        # alpha=1.0 gives the standard identity (no decay).
        self.kalman_F = self.kalman_alpha * np.eye(22)

        # B: no external control input (tweak is the state, not an increment)
        self.kalman_B = np.zeros((22, 22))

        # H: predicted Zernikes = sensitivity matrix @ tweak estimate
        # sens_mat has shape (84, 22): 4 detectors x 21 modes vs 22 active DoFs
        self.kalman_H = self.sens_mat


    def initKalmanState(self, first_tweak=None):
        """
        Initialise the Kalman filter state at the start of a simulation run.

        Builds the constant filter matrices (F, B, H) and sets the initial
        tweak estimate x and error covariance P.

        Parameters
        ----------
        first_tweak : np.ndarray, shape (22,), or None
            If provided, the tweak estimate is initialised to this value.
            In runSimulation this is set to the first OFC-computed tweak,
            which is already in the correct DoF basis and of the right
            magnitude — avoiding the large transient that arises from
            starting at zero or from the pseudo-inverse of the sensitivity
            matrix.  Default None (initialise to zero).

        Side effects
        ------------
        Calls buildKalmanMatrices().
        Sets self.kalman_x (shape 22,) and self.kalman_P (shape 22×22).
        """
        self.buildKalmanMatrices()
        if first_tweak is not None:
            self.kalman_x = first_tweak.copy()
        else:
            self.kalman_x = np.zeros(22)
        # Start with a large P so the filter trusts the first few measurements
        # heavily and converges quickly, then settles into steady-state smoothing.
        self.kalman_P = 100.0 * np.eye(22)


    def kalmanUpdate(self, z_obs_flat):
        """
        Kalman *update* step: incorporate a new Zernike measurement.

        Uses the current predicted tweak estimate (self.kalman_x, self.kalman_P)
        and the measurement noise covariance (self.kalman_R) to compute the
        posterior tweak estimate.

        Parameters
        ----------
        z_obs_flat : np.ndarray, shape (84,)
            Flattened observed Zernike residuals (4 detectors × 21 modes,
            with Z20/Z21 already stripped).  The predicted observation is
            H @ x = S @ trim.

        Side effects
        ------------
        Updates self.kalman_x and self.kalman_P in-place.
        """
        H = self.kalman_H   # (84, 22)
        R = self.kalman_R   # (84, 84)
        x = self.kalman_x   # (22,)
        P = self.kalman_P   # (22, 22)

        # Innovation: difference between observed and predicted Zernikes
        y = z_obs_flat - H @ x                  # (84,)

        # Innovation covariance
        S_inn = H @ P @ H.T + R                 # (84, 84)

        # Kalman gain
        K = P @ H.T @ np.linalg.inv(S_inn)     # (22, 84)

        # Posterior DoF state and covariance
        self.kalman_x = x + K @ y              # (22,)
        self.kalman_P = (np.eye(22) - K @ H) @ P  # (22, 22)


    def kalmanPredict(self):
        """
        Kalman *predict* step: propagate the tweak estimate forward one step.

        With F = alpha * I and B = 0:
            x_pred = alpha * x      (tweak decays toward zero)
            P_pred = alpha² * P + Q (covariance also decays, plus process noise)

        When alpha=1.0 this reduces to the original identity transition:
            x_pred = x,  P_pred = P + Q.

        Call this *after* kalmanUpdate once the current step is finalised.

        Side effects
        ------------
        Updates self.kalman_x and self.kalman_P in-place.
        """
        alpha = self.kalman_alpha
        self.kalman_x = alpha * self.kalman_x
        self.kalman_P = (alpha ** 2) * self.kalman_P + self.kalman_Q


    def kalmanFilterZernikes(self, sim_zernikes, true_tweak=None):
        """
        Apply the Kalman filter to a single noisy Zernike measurement and
        return an improved Zernike estimate in the original (4, 23) format.

        The filter state is the 22-element tweak estimate.  The measurement
        is the 84-element flattened sim_zernikes; the predicted measurement
        is H @ x = S @ tweak.  After the update step the filtered tweak
        estimate is projected back through the sensitivity matrix to produce
        filtered sim_zernikes for getTweak.

        The filter runs in two phases each call:

        1. **Update**: correct the tweak estimate using sim_zernikes as the
           noisy observation of S @ tweak.
        2. **Predict**: tweak decays by alpha, uncertainty grows by Q.

        During the burnin period (self.kalman_burnin > 0), if true_tweak is
        supplied, kalman_x is pinned to the true OFC tweak after the update.
        This lets P converge to its steady-state value before free filtering
        begins, eliminating the initial transient.  The unfiltered sim_zernikes
        is returned during burnin so getTweak receives the raw measurement.

        Parameters
        ----------
        sim_zernikes : np.ndarray, shape (4, 23)
            Noisy Zernike residual measurement (output of applyTrim in
            simulation, or wavefront sensor output on-sky).
        true_tweak : np.ndarray, shape (22,), or None
            The actual OFC-computed tweak for this step.  When provided
            and the filter is still in the burnin period, kalman_x is
            pinned to this value after the update step.

        Returns
        -------
        np.ndarray, shape (4, 23)
            During burnin: unfiltered sim_zernikes (passed through unchanged).
            After burnin:  Kalman-filtered Zernike estimate ready for getTweak.
        """
        # ── Strip Z20/Z21 (columns 16 and 17) to get the 21-mode OFC view ──
        z_21   = np.delete(sim_zernikes, [16, 17], axis=1)   # (4, 21)
        z_flat = z_21.ravel()                                  # (84,)

        # ── Update: correct tweak estimate with the Zernike measurement ──
        self.kalmanUpdate(z_flat)

        # ── Burnin: pin kalman_x to true tweak so P can converge ─────────
        if true_tweak is not None and self.kalman_burnin > 0:
            self.kalman_x = true_tweak.copy()
            self.kalman_burnin -= 1
            self.kalmanPredict()
            return sim_zernikes   # pass through unfiltered during burnin

        # ── Project filtered tweak back to Zernike space ──────────────────
        # filtered sim_zernikes = S @ tweak_filtered = H @ kalman_x
        z_filtered_flat = self.kalman_H @ self.kalman_x       # (84,)
        z_filtered_21   = z_filtered_flat.reshape(4, 21)      # (4, 21)

        # ── Reinsert Z20/Z21 as zeros (columns 16 and 17) ────────────────
        z_filtered_23 = np.insert(z_filtered_21, [16, 16], 0.0, axis=1)  # (4, 23)

        # ── Predict: tweak decays by alpha, uncertainty grows ─────────────
        self.kalmanPredict()
        return z_filtered_23


    def getTweak(self, zernikes, rotation_angle=0.0, filter_name='i',
                 subtract_intrinsics=False):
        """
        Compute the next OFC correction (tweak) for the active 22 DOFs.

        The behaviour varies depending on the control mode selected at
        construction time:

        * use_ofc_integral=True (default)
          The raw Zernikes are passed directly to the OFC.  The OFC
          internally accumulates the integral using self.ki and clamps it
          with self.max_integral.

        * use_ofc_integral=False, control_zernikes=False
          The OFC's Ki is zeroed out.  After the OFC returns a tweak in
          DOF space, an external Ki correction (self.ki * sum of the last
          n_integrals tweaks) is *subtracted* from the tweak, and the
          current tweak is appended to the history deque.

        * use_ofc_integral=False, control_zernikes=True
          Before calling the OFC, the Zernikes are pre-weighted in Zernike
          space: input = kp * zernikes + ki * sum(past_zernikes).
          The OFC's Kp is set to ones(50) and Ki to zeros(50) in
          runSimulation so the OFC applies no additional scaling.

        Parameters
        ----------
        zernikes : np.ndarray, shape (4, 23)
            Simulated Zernike coefficients at the four corner detectors.
        rotation_angle : float
            Instrument rotator angle in degrees.
        filter_name : str
            Photometric band name (e.g. 'i', 'r').
        subtract_intrinsics : bool
            If True, subtract the intrinsic Zernike offsets before computing
            the correction.

        Returns
        -------
        np.ndarray, shape (22,)
            Recommended DOF correction for the 22 active degrees of freedom.
            The tweak is always in DoF space, even when controlling
            in Vmode space.
        """
        sensor_ids = [191, 195, 199, 203]
        input_zernikes = copy.deepcopy(zernikes)
        if not self.use_ofc_integral and self.control_zernikes:
            # Apply PID in Zernike space before handing off to OFC.
            # The weighted accumulated integral is computed first, then the current
            # frame is appended to the history so it contributes to the
            # *next* step's integral (i.e. integral lags by one step).
            integral_correction = self.ki * self.evaluateIntegralTerm()
            input_zernikes = self.kp * input_zernikes + integral_correction
            self.integral_terms.append(input_zernikes)
        self.ofc.calculate_corrections(
            input_zernikes, sensor_ids, filter_name,
            rotation_angle,
            subtract_intrinsics=subtract_intrinsics,
            control_vmodes=self.control_vmodes,
        )
        tweak = self.ofc.lv_dof[self.indices]
        if not self.use_ofc_integral and not self.control_zernikes:
            # Apply external Ki correction in DOF or Vmode space.
            # Add the weighted integral of past tweaks from the current tweak,
            # then record the current tweak for future steps.
            if self.control_vmodes:
                # Convert tweak to vmodes
                full_tweak = np.zeros(50)
                full_tweak[self.indices] = tweak
                tweak_vmodes = ofc.state_estimator.get_vmodes_from_dofs(full_tweak)
                ki_correction = self.ki[0:12] * self.evaluateIntegralTerm()
                tweak_vmodes +=  ki_correction
                self.integral_terms.append(tweak_vmodes)
                # Convert back to dofs
                tweak = ofc.state_estimator.get_dofs_from_vmodes(tweak_vmodes)
            else:
                ki_correction = self.ki[self.indices] * self.evaluateIntegralTerm()
                tweak += ki_correction
                self.integral_terms.append(tweak)
        return tweak

    def extractOpenLoopReproduction(self, table):
        """
        Build the open-loop reproduction (OLR) Zernike sequence from a
        nightly observation table by removing the applied DOF trims.

        Parameters
        ----------
        table : pd.DataFrame
            Nightly AOS table loaded from parquet.  Must contain columns
            'seq', 'zk_deviation_R00/R04/R40/R44', 'dof_state',
            'rotation_angle', and 'band'.

        Returns
        -------
        tuple : (seqs, z_measured, z_opens, rots, filters)
            seqs       : list of int  – sequence numbers without NaN Zernikes
            z_measured : list of np.ndarray (4, 23) – raw measured Zernikes
            z_opens    : list of np.ndarray (4, 23) – open-loop reproduction
            rots       : list of float – rotation angles
            filters    : list of str  – band names
        """
        z_measured = []
        z_opens = []
        seqs = []
        rots = []
        filters = []
        for seq_num in range(self.seq_start, self.seq_end):
            this_table = table[table['seq'] == seq_num]
            zernikes = np.zeros([4, 23])
            zernikes[0, :] = this_table['zk_deviation_R00'].values[0]
            zernikes[1, :] = this_table['zk_deviation_R04'].values[0]
            zernikes[2, :] = this_table['zk_deviation_R40'].values[0]
            zernikes[3, :] = this_table['zk_deviation_R44'].values[0]
            if np.isnan(zernikes).any():
                print(f"{seq_num} has NaNs — skipping")
                continue
            z_measured.append(zernikes)
            # Remove applied trim to get open-loop Zernikes
            trim = this_table['dof_state'].values[0][self.indices]
            z_open = self.applyTrim(zernikes, trim, subtract=False)
            z_opens.append(z_open)
            seqs.append(seq_num)
            rots.append(this_table['rotation_angle'].values[0])
            filters.append(this_table['band'].values[0].split('_')[0])
        return seqs, z_measured, z_opens, rots, filters

    def runPIDStep(self, olr_zernikes, trim, n, stored_tweak,
                   rotation_angle=0.0, filter_name='i',
                   subtract_intrinsics=False):
        """
        Execute a single step of the PID simulation loop.

        Behaviour depends on self.discard_intermediates and
        self.smith_corrector.

        Parameters
        ----------
        olr_zernikes : np.ndarray, shape (4, 23)
            Open-loop reproduction Zernikes for this exposure.
        trim : np.ndarray, shape (22,)
            Current accumulated DOF trim.
        n : int
            Step index (0-based).
        stored_tweak : np.ndarray or deque
            When discard_intermediates=True: np.ndarray(22,) holding the
            pending tweak.
            When discard_intermediates=False: deque of length n_correct
            holding the recent tweak history.
        rotation_angle : float
            Instrument rotator angle in degrees.
        filter_name : str
            Photometric band name.
        subtract_intrinsics : bool
            Passed through to getTweak.

        Returns
        -------
        tuple : (sim_zernikes, stored_tweak, trim)
            sim_zernikes : np.ndarray (4, 23) – simulated Zernikes after trim
                (always the *raw* trimmed Zernikes, regardless of Kalman).
            stored_tweak : updated tweak storage (same type as input)
            trim         : updated accumulated trim

        Notes
        -----
        When use_kalman=True, kalmanFilterZernikes is called internally to
        produce filtered Zernikes for getTweak, and the Kalman state
        (self.kalman_x, self.kalman_P) is updated as a side effect.
        The state vector holds (z_sim_flat, trim); sim_zernikes is observed
        directly via H = [I | 0].  sim_zernikes in the return value is
        always the raw (unfiltered) measurement so that plotPID shows the
        true residual.
        """
        if self.discard_intermediates:
            if n % self.n_correct == 0:
                applied_tweak = copy.deepcopy(stored_tweak)
                trim += stored_tweak
                sim_zernikes = self.applyTrim(olr_zernikes, trim)
                tweak_input = (
                    self.kalmanFilterZernikes(sim_zernikes,
                                              true_tweak=applied_tweak)
                    if self.use_kalman else sim_zernikes
                )
                stored_tweak = self.getTweak(
                    tweak_input,
                    rotation_angle=rotation_angle,
                    filter_name=filter_name,
                    subtract_intrinsics=subtract_intrinsics,
                )
            else:
                sim_zernikes = self.applyTrim(olr_zernikes, trim)
        else:
            applied_tweak = copy.deepcopy(stored_tweak[0])
            trim += stored_tweak[0]
            sim_zernikes = self.applyTrim(olr_zernikes, trim)
            # Apply Kalman filter to the raw measurement (before Smith corrector)
            input_zernikes = (
                self.kalmanFilterZernikes(sim_zernikes,
                                          true_tweak=applied_tweak)
                if self.use_kalman else copy.deepcopy(sim_zernikes)
            )
            if self.smith_corrector:
                mod_sim_zernikes = self.sens_mat @ (stored_tweak[-1] - stored_tweak[-2])
                mod_sim_zernikes = mod_sim_zernikes.reshape(4, 21)
                # Zero out Z20 and Z21
                mod_sim_zernikes = np.insert(mod_sim_zernikes, obj=16, values=0, axis=1)
                mod_sim_zernikes = np.insert(mod_sim_zernikes, obj=16, values=0, axis=1)
                input_zernikes += mod_sim_zernikes
            stored_tweak.append(self.getTweak(
                input_zernikes,
                rotation_angle=rotation_angle,
                filter_name=filter_name,
                subtract_intrinsics=subtract_intrinsics,
            ))
        return sim_zernikes, stored_tweak, trim

    # ─────────────────────────────────────────────────────────────────────
    # Top-level run methods
    # ─────────────────────────────────────────────────────────────────────

    # ─────────────────────────────────────────────────────────────────────
    # PSF / FWHM helper methods
    # ─────────────────────────────────────────────────────────────────────

    def getPsfGradPerZernike(
        self,
        diameter: float = 8.36,
        obscuration: float = 0.612,
        jmin: int = 4,
        jmax: int = 22,
    ) -> np.ndarray:
        """
        Compute the gradient of the PSF FWHM with respect to each Zernike
        coefficient.

        Parameters
        ----------
        diameter : float, optional
            Telescope aperture diameter in metres.
            Default 8.36 m (LSST primary mirror).
        obscuration : float, optional
            Central obscuration ratio R_inner / R_outer.
            Default 0.612 (LSST primary mirror).
        jmin : int, optional
            Minimum Noll index, inclusive.  Must be >= 0.  Default 4.
        jmax : int, optional
            Maximum Noll index, inclusive.  Must be >= jmin.  Default 22.

        Returns
        -------
        np.ndarray, shape (jmax - jmin + 1,)
            dFWHM/dZ for each Noll index from jmin to jmax, in arcsec / micron.

        Raises
        ------
        ValueError
            If jmin < 0 or jmax < jmin.
        """
        if jmin < 0:
            raise ValueError("jmin cannot be negative.")
        if jmax < jmin:
            raise ValueError("jmax must be greater than jmin.")

        conversion_factors = np.zeros(jmax + 1)
        for i in range(jmin, jmax + 1):
            # Coefficient vector: all zeros up to Noll index i, then 1.
            # galsim ignores the Noll-0 placeholder at index 0.
            coefs = [0] * i + [1]
            R_outer = diameter / 2
            R_inner = R_outer * obscuration
            Z = galsim.zernike.Zernike(coefs, R_outer=R_outer, R_inner=R_inner)

            # RMS of the wavefront gradient → characteristic angular size of
            # the PSF perturbation induced by this Zernike term.
            rms_tilt = np.sqrt(np.sum(Z.gradX.coef**2 + Z.gradY.coef**2) / 2)
            rms_tilt = np.rad2deg(rms_tilt * 1e-6) * 3600  # rad → arcsec / micron
            fwhm_tilt = 2 * np.sqrt(2 * np.log(2)) * rms_tilt  # RMS → FWHM
            conversion_factors[i] = fwhm_tilt

        return conversion_factors[jmin:]

    def convertZernikesToPsfWidth(
        self,
        zernikes: np.ndarray,
        diameter: float = 8.36,
        obscuration: float = 0.612,
        jmin: int = 4,
    ) -> np.ndarray:
        """
        Convert Zernike amplitudes to their quadrature contribution to the
        PSF FWHM.

        Parameters
        ----------
        zernikes : np.ndarray
            Zernike amplitudes in microns, starting at Noll index jmin.
            Shape (N,) for a single set or (M, N) for M sets.
        diameter : float, optional
            Telescope aperture diameter in metres.  Default 8.36 m.
        obscuration : float, optional
            Central obscuration ratio R_inner / R_outer.  Default 0.612.
        jmin : int, optional
            Minimum Noll index.  Default 4 (ignores piston, tip, tilt).

        Returns
        -------
        dFWHM : np.ndarray, same shape as zernikes
            Quadrature contribution of each Zernike to the PSF FWHM,
            in arcseconds.

        Notes
        -----
        The total PSF degradation from a vector of Zernike amplitudes is
        sqrt(sum(dFWHM^2)).  This linear approximation breaks down for
        RSS(dFWHM) > ~0.20 arcsec; beyond that threshold the estimate
        tends to over-predict the PSF degradation.

        See also
        --------
        https://gist.github.com/jfcrenshaw/24056516cfa3ce0237e39507674a43e1

        Raises
        ------
        ValueError
            If jmin < 0.
        """
        if jmin < 0:
            raise ValueError("jmin cannot be negative.")
        jmax = jmin + np.array(zernikes).shape[-1] - 1
        conversion_factors = self.getPsfGradPerZernike(
            jmin=jmin, jmax=jmax,
            diameter=diameter, obscuration=obscuration,
        )
        return conversion_factors * zernikes

    def calculateAOSFWHM(self, zernikes):
        """
        Compute the AOS contribution to the PSF FWHM from a single exposure's
        Zernike measurements at the four corner detectors.

        The intrinsic (non-optical) Zernike offsets stored in ofc_data are
        subtracted before the conversion so that the returned value reflects
        only the wavefront-error component correctable by the AOS.

        Parameters
        ----------
        zernikes : np.ndarray, shape (4, 23)
            Zernike coefficients at the four corner detectors for one exposure.

        Returns
        -------
        float
            RSS (root-sum-square) of the per-Zernike FWHM contributions,
            in arcseconds.  This is the estimated AOS contribution to the
            total PSF FWHM.
        """
        intrinsics = np.array([
            self.ofc_data.y2_correction[det][0:23]
            for det in self.corner_detnames
        ])
        mean_zernikes = np.nanmean(zernikes - intrinsics, axis=0)
        zernikes_fwhm = self.convertZernikesToPsfWidth(mean_zernikes)
        return np.sqrt(np.sum(zernikes_fwhm ** 2))

    # ─────────────────────────────────────────────────────────────────────
    # Top-level run methods
    # ─────────────────────────────────────────────────────────────────────

    def buildOpenLoopReproduction(self, table):
        """
        Load the open-loop reproduction from the nightly table and store
        the results as instance attributes.

        Parameters
        ----------
        table : pd.DataFrame
            Nightly AOS table loaded from parquet.

        Side effects
        ------------
        Sets self.seqs, self.z_measured, self.z_opens, self.rots,
        self.filters.
        """
        (self.seqs, self.z_measured,
         self.z_opens, self.rots, self.filters) = self.extractOpenLoopReproduction(table)
        self.z_measured = np.array(self.z_measured)
        print(f"Built OLR: {len(self.seqs)} sequences from "
              f"{self.seq_start} to {self.seq_end}")

    def runSimulation(self):
        """
        Run the full PID simulation over all open-loop reproduction
        exposures and store the results as instance attributes.

        Requires buildOpenLoopReproduction to have been called first.

        Side effects
        ------------
        Sets self.new_zernikes, self.integrals, self.tweaks, self.trims,
        self.these_seqs.
        """
        # Push PID gains into the OFC controller
        if self.use_ofc_integral:
            self.ofc.controller.kp = self.kp
            self.ofc.controller.ki = self.ki
        else:
            if self.control_zernikes:
                # PID applied in Zernike space inside getTweak; the OFC just
                # maps the pre-weighted Zernikes to DOFs with unit gain.
                self.ofc.controller.kp = np.ones(50)
                self.ofc.controller.ki = np.zeros(50)
            else:
                self.ofc.controller.kp = self.kp
                self.ofc.controller.ki = np.zeros(50)
            self.integral_terms = []
        self.ofc.controller.kd = self.kd
        self.ofc_data.max_integral = self.max_integral
        self.ofc.controller.reset_history()

        # Initialise Kalman filter state for this run.
        # Use the first OFC-computed tweak as the initial kalman_x estimate.
        # This is well-conditioned (the OFC handles the DoF basis correctly)
        # and avoids the large transient from starting at zero or from the
        # pseudo-inverse of the sensitivity matrix.
        if self.use_kalman:
            # Restore burnin counter so re-running the simulation works correctly
            self.kalman_burnin = self._kalman_burnin_init
            first_olr = self.z_opens[0]
            first_sim = self.applyTrim(first_olr, np.zeros(22))
            first_tweak_est = self.getTweak(
                first_sim,
                rotation_angle=self.rots[0],
                filter_name=self.filters[0],
            )
            # Reset OFC history so the main loop starts clean
            self.ofc.controller.reset_history()
            self.initKalmanState(first_tweak=first_tweak_est)

        trim = np.zeros(22)
        if self.discard_intermediates:
            stored_tweak = np.zeros(22)
        else:
            stored_tweak = deque(maxlen=self.n_correct)
            for _ in range(self.n_correct):
                stored_tweak.append(np.zeros(22))

        new_zernikes = []
        integrals = []
        tweaks = []
        trims = []
        these_seqs = []
        xs = []

        for n, olr_zernikes in enumerate(self.z_opens):
            sim_zernikes, stored_tweak, trim = self.runPIDStep(
                olr_zernikes, trim, n, stored_tweak,
                rotation_angle=self.rots[n],
                filter_name=self.filters[n],
                subtract_intrinsics=False,
            )
            trims.append(copy.deepcopy(trim))
            if self.discard_intermediates:
                tweaks.append(stored_tweak)
            else:
                tweaks.append(stored_tweak[-1])
            new_zernikes.append(sim_zernikes)
            xs.append(self.kalman_x)
            if self.use_ofc_integral:
                integrals.append(self.ofc.controller.integral)
            else:
                integrals.append(np.sum(self.integral_terms, axis=0))
            these_seqs.append(self.seqs[n])

        self.new_zernikes = np.array(new_zernikes)
        self.integrals = np.array(integrals)
        self.tweaks = np.array(tweaks)
        self.trims = np.array(trims)
        self.these_seqs = these_seqs
        self.xs = np.array(xs)

    # ─────────────────────────────────────────────────────────────────────
    # Internal plotting helper
    # ─────────────────────────────────────────────────────────────────────

    def _plotAosFwhm(self, ax, plot_measured=False):
        """
        Populate a single axes with the per-exposure AOS FWHM scatter plot.

        Parameters
        ----------
        ax : matplotlib.axes.Axes
            Axes to draw on.
        plot_measured : bool
            If True, also overlay the FWHM computed from the raw measured
            Zernikes.
        """
        vals = [self.calculateAOSFWHM(x) for x in self.new_zernikes]
        ax.scatter(self.these_seqs, vals, s=20, color='tab:blue', label='AOS FWHM')
        if plot_measured:
            vals_2 = [self.calculateAOSFWHM(x) for x in self.z_measured]
            ax.scatter(self.these_seqs, vals_2, s=20, color='red',
                       marker='x', label='Measured')
        ax.set_ylabel("AOS FWHM\n[arcsec]")
        ax.set_ylim(0, 1.25)
        ax.set_yticks([0.5, 1.0])
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction="in")
        ax.legend()

    # ─────────────────────────────────────────────────────────────────────
    # Plotting methods
    # ─────────────────────────────────────────────────────────────────────

    def plotPID(self, sub_title, title, plot_measured=False, save_fig=False):
        """
        Simple summary plot: AOS FWHM and mean Zernike residuals vs.
        sequence number.

        The top row shows the AOS FWHM.  The following six rows show the
        mean (across the four corner detectors) of each Zernike group.

        Parameters
        ----------
        sub_title : str
            Descriptive subtitle added below the main figure title.
        title : str
            Output file path used when save_fig=True.
        plot_measured : bool
            If True, overlay the raw measured Zernikes / FWHM for comparison.
        save_fig : bool
            If True, save the figure to the path given by title.
        """
        fig = plt.figure(figsize=(10, 10))
        gs = gridspec.GridSpec(
            nrows=7, ncols=1,
            height_ratios=[1] * 7,
            hspace=0.0,
            wspace=0.26,
            top=0.95,
            bottom=0.05,
        )

        # Row 0: AOS FWHM
        self._plotAosFwhm(fig.add_subplot(gs[0, 0]), plot_measured=plot_measured)

        # Rows 1-6: Zernike groups
        axes = [fig.add_subplot(gs[i + 1, 0]) for i in range(6)]
        for id_group, (ax, zk_group) in enumerate(zip(axes, self.zk_groups)):
            zk_labels = self.zk_group_labels[id_group]
            for zk_idx, i in enumerate(zk_group):
                if len(zk_group) == 1:
                    zk_label = ''
                else:
                    zk_label = zk_labels.split('/')[zk_idx].strip()
                vals = np.mean(self.new_zernikes[:, :, i], axis=1)
                color = 'tab:blue' if zk_idx == 0 else 'tab:orange' if zk_idx == 1 else None
                ax.scatter(self.these_seqs, vals, s=20, color=color, label=zk_label)
                if plot_measured:
                    vals_2 = np.mean(self.z_measured[:, :, i], axis=1)
                    meas_color = 'red' if zk_idx == 0 else 'green' if zk_idx == 1 else None
                    ax.scatter(self.these_seqs, vals_2, s=20, color=meas_color,
                               marker='x', label="Meas_" + zk_label)
            ax.set_ylabel(f"{self.zk_group_labels[id_group]}\n[um]")
            if self.zk_group_labels[id_group] == 'Z4':
                ymin = -1
                ymax = 1
                ax.axhline(-0.15, ls='--', color='green')
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction="in")
            ax.set_ylim(ymin, ymax)
            ax.set_xlabel("Sequence number")
            ax.legend(bbox_to_anchor=(-0.20, 0.5), loc='upper left',
                      ncol=1, markerscale=2, frameon=False)
        my_suptitle = fig.suptitle(
            f"Simulated PID loop start={self.seq_start}\n{sub_title}",
            y=1.10, fontsize=18,
        )
        if save_fig:
            fig.savefig(
                title,
                bbox_inches='tight', pad_inches=1.2, bbox_extra_artists=[my_suptitle],
            )

    def bigPlotPID1(self, sub_title, title, plot_measured=False, save_fig=False):
        """
        Comprehensive diagnostic plot showing AOS FWHM and Zernikes at all
        four corner detectors (top panel) plus hexapod/mirror DOF trims and
        vmodes (bottom panel).

        Top panel (7 rows × 6 columns):
          Row 0       : AOS FWHM (Mean column only; remaining columns blank).
          Rows 1-6    : Zernike groups for Mean, R00, R04, R40, R44.

        Bottom panel (6 rows × 6 columns):
          Cols 0      : Hexapod DOF trims.
          Cols 1-2    : Mirror DOF trims.
          Cols 3-4    : Vmodes.

        Parameters
        ----------
        sub_title : str
            Descriptive subtitle added below the main figure title.
        title : str
            Output file path used when save_fig=True.
        plot_measured : bool
            If True, overlay the raw measured Zernikes on the Mean column.
        save_fig : bool
            If True, save the figure to the path given by title.
        """
        fig = plt.figure(figsize=(39, 22))

        gs_top = gridspec.GridSpec(
            nrows=7, ncols=6,
            width_ratios=[1] * 6,
            height_ratios=[1] * 7,
            hspace=0.0, wspace=0.38,
            top=0.97, bottom=0.54,
        )
        gs_bot = gridspec.GridSpec(
            nrows=6, ncols=6,
            width_ratios=[1] * 6,
            height_ratios=[1] * 6,
            hspace=0.0, wspace=0.38,
            top=0.46, bottom=0.05,
        )

        # Row 0: AOS FWHM in the Mean column only
        self._plotAosFwhm(fig.add_subplot(gs_top[0, 0]), plot_measured=plot_measured)

        # Rows 1-6: Zernike panels
        z_names = ['Mean', 'R00', 'R04', 'R40', 'R44']
        for j in range(5):
            axes = [fig.add_subplot(gs_top[i + 1, j]) for i in range(6)]
            for id_group, (ax, zk_group) in enumerate(zip(axes, self.zk_groups)):
                zk_labels = self.zk_group_labels[id_group]
                ymin = -1.0
                ymax = 1.0
                for zk_idx, i in enumerate(zk_group):
                    if len(zk_group) == 1:
                        zk_label = ''
                    else:
                        zk_label = zk_labels.split('/')[zk_idx].strip()
                    vals = (np.mean(self.new_zernikes[:, :, i], axis=1)
                            if j == 0 else self.new_zernikes[:, j - 1, i])
                    color = 'tab:blue' if zk_idx == 0 else 'tab:orange' if zk_idx == 1 else None
                    ax.scatter(self.these_seqs, vals, s=20, color=color, label=zk_label)
                    if plot_measured and j == 0:
                        vals_2 = np.mean(self.z_measured[:, :, i], axis=1)
                        meas_color = 'red' if zk_idx == 0 else 'green' if zk_idx == 1 else None
                        ax.scatter(self.these_seqs, vals_2, s=20, color=meas_color,
                                   marker='x', label="Meas_" + zk_label)
                ax.set_ylabel(f"{self.zk_group_labels[id_group]}\n[um]")
                if self.zk_group_labels[id_group] == 'Z4':
                    ymin = -1
                    ymax = 1
                    ax.axhline(-0.15, ls='--', color='green')
                ax.grid(True, alpha=0.5)
                ax.tick_params(direction="in")
                ax.set_ylim(ymin, ymax)
                ax.set_xlabel("Sequence number")
                if id_group == 0:
                    ax.set_title(z_names[j])
                ax.legend(bbox_to_anchor=(-0.20, 0.5), loc='upper left',
                          ncol=1, markerscale=2, frameon=False)

        # Hexapod DOF trims
        axes = [fig.add_subplot(gs_bot[i, 0]) for i in range(6)]
        for id_group, (ax, dof_group) in enumerate(zip(axes, self.groups)):
            for i in dof_group:
                vals = self.trims[:, i]
                plot_vals = 3600.0 * vals if i in [3, 4, 8, 9] else vals
                ax.scatter(self.these_seqs, plot_vals, label=f"{self.labels[i]}", s=3)
            ax.set_ylabel(self.group_labels[id_group])
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction="in")
            leg = ax.legend(bbox_to_anchor=(1.28, 0.5), loc='center right', markerscale=3)
            leg.get_frame().set_linewidth(0)
            leg.get_frame().set_edgecolor("none")
            leg.get_frame().set_facecolor("none")
        for ax in axes[:-1]:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        axes[0].set_title('Hexapods State (Trim only)')
        axes[-1].set_xlabel("Sequence Number")

        # Mirror DOF trims
        axes1 = [fig.add_subplot(gs_bot[i, 1]) for i in range(6)]
        axes2 = [fig.add_subplot(gs_bot[i, 2]) for i in range(6)]
        mirror_axes = axes1 + axes2
        mirror_indices = list(range(10, 17)) + list(range(30, 35))
        for n, i in enumerate(mirror_indices):
            mirror_axes[n].scatter(self.these_seqs, self.trims[:, n], s=11)
            mirror_axes[n].set_ylabel(self.all_labels[i])
        for ax in mirror_axes:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        axes1[0].set_title('Mirror DOFs (Trim only)')
        axes1[5].tick_params(labelbottom=True)
        axes1[5].set_xlabel("Sequence Number")
        axes2[0].set_title('Mirror DOFs (Trim only)')
        axes2[5].tick_params(labelbottom=True)
        axes2[5].set_xlabel("Sequence Number")

        # Vmodes
        vmodes = []
        full_indices = list(range(0, 17)) + list(range(30, 35))
        for i in range(self.trims.shape[0]):
            this_trim = np.zeros(50)
            this_trim[full_indices] = self.trims[i, :]
            vmodes.append(self.state_estimator.get_vmodes_from_dofs(this_trim))
        vmodes = np.array(vmodes)

        vaxes1 = [fig.add_subplot(gs_bot[i, 3]) for i in range(6)]
        vaxes2 = [fig.add_subplot(gs_bot[i, 4]) for i in range(6)]
        vaxes = vaxes1 + vaxes2
        for i in range(12):
            vaxes[i].scatter(self.these_seqs, vmodes[:, i], s=11)
            vaxes[i].set_ylabel(self.vmode_labels[i])
        for ax in vaxes:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        vaxes1[0].set_title('Vmodes')
        vaxes1[5].tick_params(labelbottom=True)
        vaxes1[5].set_xlabel("Sequence Number")
        vaxes2[0].set_title('Vmodes')
        vaxes2[5].tick_params(labelbottom=True)
        vaxes2[5].set_xlabel("Sequence Number")

        my_suptitle = fig.suptitle(
            f"Simulated PID loop start={self.seq_start}\n{sub_title}",
            y=1.02, fontsize=18,
        )
        if save_fig:
            fig.savefig(
                title,
                bbox_inches='tight', pad_inches=1.2, bbox_extra_artists=[my_suptitle],
            )

    def bigPlotPID2(self, sub_title, title, plot_measured=False, save_fig=False):
        """
        Alternative diagnostic plot showing AOS FWHM, mean Zernike residuals,
        and integral terms (top panel) plus hexapod/mirror DOF trims and
        vmodes (bottom panel).

        Top panel (7 rows × 6 columns):
          Row 0, Col 0 : AOS FWHM.
          Rows 1-6, Col 0 : Mean Zernike residuals per group.
          Rows 0-5, Cols 1-4 : Integral terms (up to 24 shown).

        Bottom panel (6 rows × 6 columns):
          Cols 0      : Hexapod DOF trims.
          Cols 1-2    : Mirror DOF trims.
          Cols 3-4    : Vmodes.

        Parameters
        ----------
        sub_title : str
            Descriptive subtitle added below the main figure title.
        title : str
            Output file path used when save_fig=True.
        plot_measured : bool
            If True, overlay the raw measured Zernikes for comparison.
        save_fig : bool
            If True, save the figure to the path given by title.
        """
        fig = plt.figure(figsize=(39, 22))

        gs_top = gridspec.GridSpec(
            nrows=7, ncols=6,
            width_ratios=[1] * 6,
            height_ratios=[1] * 7,
            hspace=0.0, wspace=0.38,
            top=0.97, bottom=0.54,
        )
        gs_bot = gridspec.GridSpec(
            nrows=6, ncols=6,
            width_ratios=[1] * 6,
            height_ratios=[1] * 6,
            hspace=0.0, wspace=0.38,
            top=0.46, bottom=0.05,
        )

        # Row 0, Col 0: AOS FWHM
        self._plotAosFwhm(fig.add_subplot(gs_top[0, 0]), plot_measured=plot_measured)

        # Rows 1-6, Col 0: mean Zernike residuals
        zk_axes = [fig.add_subplot(gs_top[i + 1, 0]) for i in range(6)]
        zk_axes[0].set_title("Zernikes")
        for id_group, (ax, zk_group) in enumerate(zip(zk_axes, self.zk_groups)):
            zk_labels = self.zk_group_labels[id_group]
            ymin = -1.0
            ymax = 1.0
            for zk_idx, i in enumerate(zk_group):
                if len(zk_group) == 1:
                    zk_label = ''
                else:
                    zk_label = zk_labels.split('/')[zk_idx].strip()
                vals = np.mean(self.new_zernikes[:, :, i], axis=1)
                color = 'tab:blue' if zk_idx == 0 else 'tab:orange' if zk_idx == 1 else None
                ax.scatter(self.these_seqs, vals, s=20, color=color, label=zk_label)
                if plot_measured:
                    vals_2 = np.mean(self.z_measured[:, :, i], axis=1)
                    meas_color = 'red' if zk_idx == 0 else 'green' if zk_idx == 1 else None
                    ax.scatter(self.these_seqs, vals_2, s=20, color=meas_color,
                               marker='x', label="Meas_" + zk_label)
            ax.set_ylabel(f"{self.zk_group_labels[id_group]}\n[um]")
            if self.zk_group_labels[id_group] == 'Z4':
                ymin = -1
                ymax = 1
                ax.axhline(-0.15, ls='--', color='green')
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction="in")
            ax.set_ylim(ymin, ymax)
            ax.set_xlabel("Sequence number")
            ax.legend(bbox_to_anchor=(-0.20, 0.5), loc='upper left',
                      ncol=1, markerscale=2, frameon=False)

        # Rows 0-5, Cols 1-4: integral terms (6 rows × 4 cols = 24 slots)
        integral_index = 0
        int_indices = list(range(0, 17)) + list(range(30, 35))
        for j in range(1, 5):
            int_axes = [fig.add_subplot(gs_top[i, j]) for i in range(7)]
            int_axes[0].set_title("Integral terms")
            for ax in int_axes:
                if self.control_vmodes and integral_index > 11:
                    ax.set_visible(False)
                    continue
                if integral_index > 21:
                    ax.set_visible(False)
                    continue
                if self.control_zernikes:
                    vals = np.mean(self.integrals[:, :, integral_index], axis=1)
                    int_label = f"Z{integral_index + 4}"
                else:
                    vals = self.integrals[:, integral_index]
                    if self.control_vmodes:
                        int_label = f"Vmode{integral_index + 1}"
                    else:
                        int_label = self.all_labels[int_indices[integral_index]]
                ax.scatter(self.these_seqs, vals, s=20, color='tab:blue')
                ax.set_ylabel(int_label)
                ax.grid(True, alpha=0.5)
                ax.tick_params(direction="in")
                ax.set_xlabel("Sequence number")
                integral_index += 1

        # Hexapod DOF trims
        axes = [fig.add_subplot(gs_bot[i, 0]) for i in range(6)]
        for id_group, (ax, dof_group) in enumerate(zip(axes, self.groups)):
            for i in dof_group:
                vals = self.trims[:, i]
                plot_vals = 3600.0 * vals if i in [3, 4, 8, 9] else vals
                ax.scatter(self.these_seqs, plot_vals, label=f"{self.labels[i]}", s=3)
            ax.set_ylabel(self.group_labels[id_group])
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction="in")
            leg = ax.legend(bbox_to_anchor=(1.28, 0.5), loc='center right', markerscale=3)
            leg.get_frame().set_linewidth(0)
            leg.get_frame().set_edgecolor("none")
            leg.get_frame().set_facecolor("none")
        for ax in axes[:-1]:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        axes[0].set_title('Hexapods State (Trim only)')
        axes[-1].set_xlabel("Sequence Number")

        # Mirror DOF trims
        axes1 = [fig.add_subplot(gs_bot[i, 1]) for i in range(6)]
        axes2 = [fig.add_subplot(gs_bot[i, 2]) for i in range(6)]
        mirror_axes = axes1 + axes2
        mirror_indices = list(range(10, 17)) + list(range(30, 35))
        for n, i in enumerate(mirror_indices):
            mirror_axes[n].scatter(self.these_seqs, self.trims[:, n], s=11)
            mirror_axes[n].set_ylabel(self.all_labels[i])
        for ax in mirror_axes:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        axes1[0].set_title('Mirror DOFs (Trim only)')
        axes1[5].tick_params(labelbottom=True)
        axes1[5].set_xlabel("Sequence Number")
        axes2[0].set_title('Mirror DOFs (Trim only)')
        axes2[5].tick_params(labelbottom=True)
        axes2[5].set_xlabel("Sequence Number")

        # Vmodes
        vmodes = []
        full_indices = list(range(0, 17)) + list(range(30, 35))
        for i in range(self.trims.shape[0]):
            this_trim = np.zeros(50)
            this_trim[full_indices] = self.trims[i, :]
            vmodes.append(self.state_estimator.get_vmodes_from_dofs(this_trim))
        vmodes = np.array(vmodes)

        vaxes1 = [fig.add_subplot(gs_bot[i, 3]) for i in range(6)]
        vaxes2 = [fig.add_subplot(gs_bot[i, 4]) for i in range(6)]
        vaxes = vaxes1 + vaxes2
        for i in range(12):
            vaxes[i].scatter(self.these_seqs, vmodes[:, i], s=11)
            vaxes[i].set_ylabel(self.vmode_labels[i])
        for ax in vaxes:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        vaxes1[0].set_title('Vmodes')
        vaxes1[5].tick_params(labelbottom=True)
        vaxes1[5].set_xlabel("Sequence Number")
        vaxes2[0].set_title('Vmodes')
        vaxes2[5].tick_params(labelbottom=True)
        vaxes2[5].set_xlabel("Sequence Number")

        my_suptitle = fig.suptitle(
            f"Simulated PID loop start={self.seq_start}\n{sub_title}",
            y=1.02, fontsize=18,
        )
        if save_fig:
            fig.savefig(
                title,
                bbox_inches='tight', pad_inches=1.2, bbox_extra_artists=[my_suptitle],
            )


## Build OFC, state estimator and sensitivity matrix
You will need to `git clone ts_config_mttcs` and replace the config path below.

In [ ]:
ofc_data = OFCData(
    name="lsst",
    config_dir="/home/c/cslage/WORK/ts_config_mttcs/MTAOS/ofc",
)

ofc = OFC(ofc_data=ofc_data)

new_comp_dof_idx = dict(
    m2HexPos=np.ones(5, dtype=bool),
    camHexPos=np.ones(5, dtype=bool),
    M1M3Bend=np.ones(20, dtype=bool),
    M2Bend=np.ones(20, dtype=bool),
)
new_comp_dof_idx["M1M3Bend"][7:] = False
new_comp_dof_idx["M2Bend"][5:] = False

ofc.set_truncation_index(12)
ofc_data.zn_selected = np.array([4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26])
ofc.ofc_data.comp_dof_idx = new_comp_dof_idx
ofc.controller.reset_history()
ofc.state_estimator.refresh_from_ofc_data()

corner_detnames = ["R00_SW0", "R04_SW0", "R40_SW0", "R44_SW0"]
field_angles_CCS = [ofc_data.sample_points[det] for det in corner_detnames]

state_estimator = StateEstimator(ofc_data)
sens_mat = state_estimator.get_sensitivity_matrix(field_angles_CCS, 0.0)

## Set PID gains and instantiate the simulation
Edit the gain arrays and parameters below to configure the run.

In [ ]:
# DOF index: collapses the full 50-DOF space down to the 22 DOFs we are using.
# If control_vmodes=True the _used arrays should have 12 components.
# If control_vmodes=False they should have 22 components.
indices = list(range(0, 17)) + list(range(30, 35))



# ── Starting point for control_vmodes=False ───────────────────────────
kp = np.zeros(50)
kp_used = 0.3 * np.ones(22)
kp[indices] = kp_used
#kp[0] = 0.05; kp[5] = 0.05
ki = np.zeros(50)
ki_used = 0.0 * np.ones(22)
ki[indices] = ki_used
ki[0] = 0.2; ki[5] = 0.2

kd = np.zeros(50)

"""

# ── Starting point for control_vmodes=True ────────────────────────────
kp = np.zeros(50)
kp_used = np.array([0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3])
kp[0:12] = kp_used
ki = np.zeros(50)
ki_used = np.array([0.0, 0.0, 0.0, 0.0, 0.00, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
ki[0:12] = ki_used
kd = np.zeros(50)




# Starting point when control_zernikes=True
kd = np.zeros(50)
kp = 0.3 * np.ones([4,23])
ki = np.zeros([4,23])
for i in range(4):
    pass
    kp[i,0] = 0.2
    ki[i,0] = 0.2

"""

# Integral term options. Only used when use_ofc_integral=True
# ── Integral clamp (max absolute value the integral term can reach) ───
max_integral = np.zeros(50)
max_integral_used = 5000.0 * np.ones(22)
max_integral[indices] = max_integral_used

In [ ]:
filename = "/home/c/cslage/u/MTAOS/data/kalman.pkl"
with open(filename, 'rb') as f:
    kalman_dict = pkl.load(f)
kalman_r = kalman_dict['R']

## Notes on Ki options:
(1) use_ofc_integral=True\
Integral term for each DoF will be kept inside ofc and trimmed using max_integral.\
\
(2) use_ofc_integral=False and control_zernikes=False\
Integral term for each DoF will be kept in the notebook.\
No trimming, each DoF will keep the integral for n_integral terms.\
\
(3) use_ofc_integral=False and control_zernikes=True\
Kp and Ki terms will be applied in Zernike space.\
Each Zernike coefficient will keep the integral for n_integral terms.\

In [ ]:
sim = PIDLoopSimulation(
    ofc=ofc,
    ofc_data=ofc_data,
    state_estimator=state_estimator,
    sens_mat=sens_mat,
    kp=kp,
    ki=ki,
    kd=kd,
    day_obs=20260329,
    seq_start=4,
    seq_end=200,
    n_correct=3,
    discard_intermediates=True,
    smith_corrector=False,
    control_vmodes=False,
    use_ofc_integral=False,
    max_integral=max_integral,
    control_zernikes=False,
    leaky_integral_factor=0.5,
    # ── Kalman filter (optional) ───────────────────────────────────────────
    use_kalman=True,         # set True to enable
    kalman_r_sigma=1E-6,      # measurement noise std dev (microns)
    kalman_q_dof_sigma=0.01,  # process noise on DoF state
    kalman_R_matrix=kalman_r,     # supply (84,84) array to override kalman_r_sigma
    kalman_r_floor=1e-6,          # floor on R diagonal to prevent singularity
    kalman_alpha=0.9,             # leaky factor: 1.0=no decay, try 0.8-0.99
    kalman_burnin=10,             # steps to pin kalman_x to true tweak before filtering
)

## Load the nightly parquet table
This parquet file must be extracted at the summit — USDF software is behind!

The parquet file was created with:
https://github.com/lsst-sitcom/ts_aos_analysis/blob/tickets/DM-54406/notebooks/nightly_report/nightly_report_ts_version.ipynb

In [ ]:
parquet_file = f"/home/c/cslage/u/MTAOS/times_square_reports/nightly_aos_table_{sim.day_obs}_summit.parquet"
table = pd.read_parquet(parquet_file)
print(f'Loaded {parquet_file}: {len(table)} rows')
print(f'Columns: {sorted(table.columns.tolist())}')

## Build the open-loop reproduction

In [ ]:
sim.buildOpenLoopReproduction(table)

## Run the simulation
Integral terms, trims, and tweaks are stored on the `sim` object for later plotting.

In [ ]:
sim.runSimulation()

## Plot the results
Three plotting options are available:
- `sim.plotPID(...)` — mean Zernike residuals only
- `sim.bigPlotPID1(...)` — Zernikes at all 4 corners + DOFs + vmodes
- `sim.bigPlotPID2(...)` — integral terms + DOFs + vmodes

In [ ]:
meas_z4 = []
sim_z4 = []
meas_aos = []
sim_aos = []
for i, seq in enumerate(sim.these_seqs):
    #if seq < 300 or seq > 600:
    if seq < 25:
        continue
    meas_z4.append(np.mean(sim.z_measured[i, :, 0]))
    sim_z4.append(np.mean(sim.new_zernikes[i, :, 0]))
    meas_aos.append(sim.calculateAOSFWHM(sim.z_measured[i]))
    sim_aos.append(sim.calculateAOSFWHM(sim.new_zernikes[i]))

sub = f"{sim.day_obs}, Kp=0.3, Ki=0.0, discard intermediates\n Ncorr=3, DoFs, Ifactor=0.5, Kalman, Q=1000.0"
sub += f"\nMeas Z4: Median = {np.median(meas_z4):.3f}, Std = {np.std(meas_z4):.3f}, New_sim Z4: Median = {np.median(sim_z4):.3f}, Std = {np.std(sim_z4):.3f}"
sub += f"\nMeas AOS_FWHM: Median = {np.median(meas_aos):.3f}, Std = {np.std(meas_aos):.3f}, New_sim AOS_FWHM: Median = {np.median(sim_aos):.3f}, Std = {np.std(sim_aos):.3f}"
sim.plotPID(sub, f"/home/c/cslage/u/MTAOS/pid_output/PID_Simulator_New_Kalman_3_{sim.day_obs}.png", plot_measured=True, save_fig=True)

In [ ]:
sub = f"{sim.day_obs}, Kp=0.3, Ki=0.0,Ki(0,5)=0.2, discard intermediates\n Ncorr=3, DoFs, Ifactor=0.5"
sim.plotPID(sub, f"/home/c/cslage/u/MTAOS/pid_output/PID_Simulator_N28_{sim.day_obs}.png", plot_measured=True, save_fig=True)

In [ ]:
sub = f"{sim.day_obs}, Kp=0.3, Ki=0.0, discard intermediates, Vmodes"
sim.bigPlotPID1(sub, f"PID_Simulator_Test_{sim.day_obs}")
# sim.bigPlotPID1(sub, f"...", plot_measured=True)

In [ ]:
sub = f"{sim.day_obs}, Kp=0.3,Kp(10)=0.05, Ki=0.0,Ki(10)=0.5, discard intermediates, Vmodes, integrate_dofs, local_integrals"
sim.bigPlotPID2(sub, f"/home/c/cslage/u/MTAOS/pid_output/PID_Simulator_N18_{sim.day_obs}.png", save_fig=True)
# sim.bigPlotPID2(sub, f"...", plot_measured=True)

In [ ]:
def specialPlot(sim, sub_title, title, plot_measured=False, save_fig=False):
    """
    Alternative diagnostic plot showing mean Zernike residuals and
    integral terms (top panel) plus hexapod/mirror DOF trims and
    vmodes (bottom panel).

    Parameters
    ----------
    sub_title : str
        Descriptive subtitle added below the main figure title.
    title : str
        Base filename (without extension) used when saving the figure.
    plot_measured : bool
        If True, overlay the raw measured Zernikes for comparison.
    """
    fig = plt.figure(figsize=(10,10))

    gs = gridspec.GridSpec(
        nrows=5, ncols=2,
        width_ratios=[1] * 2,
        height_ratios=[1] * 5,
        hspace=0.0, wspace=0.38,
        top=0.97, bottom=0.05,
    )

    # Zernike column
    # Row 0: AOS FWHM
    sim._plotAosFwhm(fig.add_subplot(gs[0, 0]), plot_measured=plot_measured)

    # Rows 1-6: selected Zernikes and Hexapods
    axes = [fig.add_subplot(gs[i, 0]) for i in range(1,5)]
    for ax in axes:
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction="in")
    vals_0 = np.mean(sim.new_zernikes[:, :, 0], axis=1)
    axes[0].scatter(sim.these_seqs, vals_0, s=20, color='tab:blue', label='Sim')
    axes[0].set_ylim(-1.0, 1.0)
    axes[0].set_ylabel('Z4')
    axes[0].axhline(-0.2, ls='--', color='k')
    if plot_measured:
        vals_m0 = np.mean(sim.z_measured[:, :, 0], axis=1)
        axes[0].scatter(sim.these_seqs, vals_m0, s=20, color='red',
                   marker='x', label="Meas")
    axes[0].legend()
    vals_1 = np.mean(sim.new_zernikes[:, :, 7], axis=1)
    axes[1].scatter(sim.these_seqs, vals_1, s=20, color='tab:blue', label='Sim')
    axes[1].set_ylim(-0.2, 0.2)
    axes[1].set_ylabel('Z11')
    axes[1].axhline(0.0, ls='--', color='k')
    axes[1].legend()
    if plot_measured:
        vals_m1 = np.mean(sim.z_measured[:, :, 7], axis=1)
        axes[1].scatter(sim.these_seqs, vals_m1, s=20, color='red',
                   marker='x', label="Meas")
    axes[1].legend()
    vals_2 = sim.trims[:, 0]
    axes[2].scatter(sim.these_seqs, vals_2, s=20, color='tab:blue')
    #axes[2].set_ylim(-100, 200)
    axes[2].set_ylabel('M2Z')
    vals_3 = sim.trims[:, 5]
    axes[3].scatter(sim.these_seqs, vals_3, s=20, color='tab:blue')
    #axes[3].set_ylim(100,400)
    axes[3].set_ylabel('CamZ')
    axes[3].set_xlabel("Sequence number")

    if sim.control_zernikes:
        int_list = [0, 1, 2, 7]
    else:
        if sim.control_vmodes:
            int_list = [4, 9]
        else:
            int_list = [0, 5]
    int_axes = [fig.add_subplot(gs[i, 1]) for i in range(len(int_list))]
    int_axes[0].set_title("Integral terms")
    for i, ax in enumerate(int_axes):
        integral_index = int_list[i]
        if sim.control_zernikes:
            vals = np.mean(sim.integrals[:, :, integral_index], axis=1)
            int_label = f"Z{integral_index + 4}"
        else:
            vals = sim.integrals[:, integral_index]
            if sim.control_vmodes:
                int_label = sim.vmode_labels[int_list[i]]
            else:
                int_label = sim.all_labels[int_list[i]]
        ax.scatter(sim.these_seqs, vals, s=20, color='tab:blue')
        ax.set_ylabel(int_label)
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction="in")
        ax.set_xlabel("Sequence number")
        integral_index += 1

    
    my_suptitle = fig.suptitle(
        f"Simulated PID loop \n{sub_title}",
        y=1.10, fontsize=18,
    )
    if save_fig:
        fig.savefig(
        title,
        bbox_inches='tight', pad_inches=1.2, bbox_extra_artists=[my_suptitle],
    )


In [ ]:
sub = f"{sim.day_obs}, Kp=0.3,Kp(10)=0.05, Ki=0.0,Ki(10)=0.5\ndiscard intermediates, Vmodes\nintegrate_dofs, local_integrals"
specialPlot(sim, sub, f"/home/c/cslage/u/MTAOS/pid_output/Special_N25_{sim.day_obs}.png", plot_measured=True, save_fig=False)

In [ ]:
def filterPlot(sim, sub_title, title, save_fig=False):
    fig = plt.figure(figsize=(20,10))
    gs = gridspec.GridSpec(
        nrows=6, ncols=4,
        width_ratios=[1] * 4,
        height_ratios=[1] * 6,
        hspace=0.0, wspace=1.0,
        top=0.95, bottom=0.05,
    )

    axes = [fig.add_subplot(gs[i, 0]) for i in range(6)]
    for id_group, (ax, dof_group) in enumerate(zip(axes, sim.groups)):
        for i in dof_group:
            vals = sim.tweaks[:, i]
            vals_2 = sim.xs[:, i]
            plot_vals = 3600.0 * vals if i in [3, 4, 8, 9] else vals
            plot_vals_2 = 3600.0 * vals_2 if i in [3, 4, 8, 9] else vals_2
            ax.scatter(sim.these_seqs, plot_vals, label="Tweak", s=3)
            ax.scatter(sim.these_seqs, plot_vals_2, label="Kalman x", s=3)
        ax.set_ylabel(sim.group_labels[id_group])
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction="in")
        leg = ax.legend(bbox_to_anchor=(1.50, 0.5), loc='center right', markerscale=3)
        leg.get_frame().set_linewidth(0)
        leg.get_frame().set_edgecolor("none")
        leg.get_frame().set_facecolor("none")
    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)
        ax.grid(True, alpha=0.5)
    axes[0].set_title('Hexapods State (Tweaks)')
    axes[-1].set_xlabel("Sequence Number")

    # Mirror DOF trims
    axes1 = [fig.add_subplot(gs[i, 1]) for i in range(6)]
    axes2 = [fig.add_subplot(gs[i, 2]) for i in range(6)]
    mirror_axes = axes1 + axes2
    mirror_indices = list(range(10, 17)) + list(range(30, 35))
    for n, i in enumerate(mirror_indices):
        mirror_axes[n].scatter(sim.these_seqs, sim.tweaks[:, n], s=11, label="Tweak")
        mirror_axes[n].scatter(sim.these_seqs, sim.xs[:, n], s=11, label="Kalman_x")
        mirror_axes[n].set_ylabel(sim.all_labels[i])
        leg = mirror_axes[n].legend(bbox_to_anchor=(1.60, 0.5), loc='center right', markerscale=2)
        leg.get_frame().set_linewidth(0)
        leg.get_frame().set_edgecolor("none")
        leg.get_frame().set_facecolor("none")
    for ax in mirror_axes:
        ax.tick_params(labelbottom=False)
        ax.grid(True, alpha=0.5)
    axes1[0].set_title('Mirror DOFs (Tweaks)')
    axes1[5].tick_params(labelbottom=True)
    axes1[5].set_xlabel("Sequence Number")
    axes2[0].set_title('Mirror DOFs (Tweaks)')
    axes2[5].tick_params(labelbottom=True)
    axes2[5].set_xlabel("Sequence Number")

    my_suptitle = fig.suptitle(
        f"Simulated PID loop start={sim.seq_start}\n{sub_title}",
        y=1.10, fontsize=18,
    )
    if save_fig:
        fig.savefig(
            title,
            bbox_inches='tight', pad_inches=1.2, bbox_extra_artists=[my_suptitle],
        )


In [ ]:
sub = f"{sim.day_obs}, Kp=0.3, Ki=0.0, discard intermediates\n Ncorr=3, DoFs, Ifactor=0.5, Kalman, Q=10, alpha=0.9, Burn-in=10"
filterPlot(sim, sub, f"/home/c/cslage/u/MTAOS/pid_output/Kalman_X_12_{sim.day_obs}.png", save_fig=True)